# Genetic toggle switch: bistability

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZetaOne/pyauto-xpp/blob/main/tutorials/notebooks/02_toggle_switch.ipynb)

Two genes repress each other (Gardner, Cantor & Collins 2000). With strong enough
repression there are two stable states (one gene on, the other off) separated by a saddle;
which one the cell ends in depends on where it starts. Weakening one gene's synthesis
`a1` removes one state in a fold (saddle-node).

*Based on an example from [XPP Studio](https://xpp-studio.github.io/xpp-studio/) (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.4 and its AUTO through pyauto-xpp.*

Running in Google Colab? This cell installs pyauto-xpp first; elsewhere it does nothing.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip -q install pyauto-xpp "plotly>=6" anywidget ipywidgets --find-links https://github.com/ZetaOne/pyauto-xpp/releases/expanded_assets/v0.5.0
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ZetaOne/pyauto-xpp/main/tutorials/notebooks/xpptut.py", "xpptut.py")
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [3]:
mb = (ModelBuilder("toggle_switch")
      .comment("Genetic toggle switch (Gardner, Cantor & Collins 2000)")
      .ode("u", "a1/(1+v^beta)-u")
      .ode("v", "a2/(1+u^gamma)-v")
      .parameter(a1=3, a2=3, beta=2, gamma=2)
      .initial(u=2.5, v=0.4)
      .option(total=30, dt=0.05))
print(mb.render())

# Genetic toggle switch (Gardner, Cantor & Collins 2000)
u'=a1/(1+v^beta)-u
v'=a2/(1+u^gamma)-v
par a1=3,a2=3,beta=2,gamma=2
init u=2.5,v=0.4
@ total=30,dt=0.05
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [4]:
ex = Explorer(mb, kind="plane", x="u", y="v", window=(-0.2, 3.6, -0.2, 3.6),
              sliders={"a1": (0.5, 6), "a2": (0.5, 6), "beta": (1, 4), "gamma": (1, 4)},
              auto="a1", auto_range=(0.5, 6), auto2="a2", auto2_range=(0.5, 6), title="Toggle switch")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**Which state wins?** Integrate from a grid of starting points and colour each by where
it ends: the boundary between the two basins is the saddle's stable manifold.

In [5]:
import numpy as np
U0, V0 = np.meshgrid(np.linspace(0, 3.5, 25), np.linspace(0, 3.5, 25))
end = np.array([[ex.m.run(keep=False, u=a, v=b, total=40)["u"][-1] for a in U0[0]] for b in V0[:, 0]])
show_figure(go.Figure(go.Heatmap(x=U0[0], y=V0[:, 0], z=(end > 1).astype(int), colorscale=[[0, "#bfdbfe"], [1, "#fecaca"]],
                                 showscale=False)), xaxis_title="u(0)", yaxis_title="v(0)", height=380)

**The fold.** AUTO follows the equilibria in `a1`; the two LP points bound the bistable range.

In [6]:
parts = ex.run_auto()
print(sorted({round(p.x, 4) for d in parts for p in d.special("LP")}))

[2.7008, 3.5424]


## Things to try

- Lower `beta` and `gamma` towards 1: the bistability disappears (cooperativity is needed).
- Use the AUTO panel: click an LP point and get its two-parameter curve in (a1, a2) — the bistable wedge.